# MLP

### Import requirements

In [ ]:
# Required imports
# --------------------------------------------------------------------------
## Basic libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
import time
import pickle
import joblib

#----------------------------------------------------------------------------
## Statistical functions
from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy.stats import norm

# ---------------------------------------------------------------------------
## Scikit-learn imports
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.base import BaseEstimator, TransformerMixin, clone, ClassifierMixin

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import BayesianRidge, LogisticRegression
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GridSearchCV, cross_validate
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss, precision_recall_curve, average_precision_score, log_loss, accuracy_score, precision_score, recall_score, f1_score
from sklearn.calibration import calibration_curve
from sklearn.utils.validation import check_is_fitted
from sklearn.feature_selection import RFE

from sklearn.inspection import permutation_importance

# --------------------------------------------------------------------------
## Scikit-optimize
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

#---------------------------------------------------------------------------
## SHAP
import shap

In [ ]:
# Helper functions and custom classes

#-----------------------------------------------------------------------------
## Performance evaluation

### Calculatiion of calibration intercept and slope using logistic model
def calibration_intercept_slope(
    y_true,
    p_pred,
    epsilon=1e-6,
    max_iter=5000,
    alpha=0.05
):
    """
    Takes y_true (actual outcomes) and predicted probabilities as inputs
    Calculates calibration intercept and slope with 95% CI
    """
    # Format y_true and predicted probabilities as np arrays
    y_true = np.asarray(y_true).astype(int) # only 0s and 1s
    p = np.clip(np.asarray(p_pred), epsilon, 1 - epsilon) # handle p=0 or p=1

    # Logit of predicted probabilities
    logit_p = np.log(p / (1 - p))

    # Construct a dataframe for the logistic regression
    df = pd.DataFrame({
        "y": y_true,
        "logit_p": logit_p
    })

    # Fit logistic regression
    calib_model = smf.logit("y ~ logit_p", data=df) 
    result = calib_model.fit(disp=False, maxiter=max_iter)

    # Obtain point estimates
    intercept = float(result.params["Intercept"])
    slope = float(result.params["logit_p"])

    # Confidence intervals
    ci = result.conf_int(alpha=alpha)
    intercept_ci = tuple(ci.loc["Intercept"].astype(float))
    slope_ci = tuple(ci.loc["logit_p"].astype(float))

    return {
        "intercept": intercept,
        "intercept_ci_95": intercept_ci,
        "slope": slope,
        "slope_ci_95": slope_ci,
    }


# Calculation of O/E ratio (simple division)
def observed_expected_ratio(y_true, p_pred):
    """
    O/E ratio = sum(y) / sum(p)
    """
    y_true = np.asarray(y_true).astype(float)
    p_pred = np.asarray(p_pred).astype(float)
    
    return float(y_true.sum() / p_pred.sum())
    

# Calculation of O/E ratio with 95% confidence intervals (grouped bootstrap)
def oe_ratio_ci_cluster_bootstrap(y_true, p_pred, groups, B=5000, alpha=0.05, seed=0):
    """
    Cluster bootstrap CI for O/E ratio.
    Returns: (point_estimate, ci_low, ci_high)
    Example use:
    oe, lo, hi = observed_expected_ratio_grouped_bootstrap(y_true, p_pred, groups, B=10000, seed=12)
    """
    y_true = np.asarray(y_true)
    p_pred = np.asarray(p_pred)
    groups = np.asarray(groups)

    # Point estimate on the full sample
    point = observed_expected_ratio(y_true, p_pred)

    # Prepare group -> indices mapping
    unique_groups = np.unique(groups)
    idx_by_g = {g: np.where(groups == g)[0] for g in unique_groups}

    rng = np.random.default_rng(seed)
    boot = np.full(B, np.nan, dtype=float)
    
    for i in range(B):
        sampled_groups = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        idx = np.concatenate([idx_by_g[g] for g in sampled_groups])  # duplicates groups properly
        boot[i] = observed_expected_ratio(y_true[idx], p_pred[idx])

    # Percentile CI
    ci_low, ci_high = np.nanquantile(boot, [alpha/2, 1 - alpha/2])

    return {
            "oe_ratio": float(point),
            "oe_ratio_ci_95": (float(ci_low), float(ci_high))
        }


def compute_metrics(y_true, p, thr=0.5):
    """
    Example use:
    pooled = compute_metrics(oof_true, oof_proba, thr=0.5)
    pooled
    """

    y_hat = (p >= thr).astype(int)
    return {
        "AUROC": roc_auc_score(y_true, p),
        "Brier": brier_score_loss(y_true, p),
        "LogLoss": log_loss(y_true, p),
        "Accuracy": accuracy_score(y_true, y_hat),
        "Precision": precision_score(y_true, y_hat, zero_division=0),
        "Recall": recall_score(y_true, y_hat, zero_division=0),
        "F1": f1_score(y_true, y_hat, zero_division=0),
        "Average precision": average_precision_score(y_true, p)
    }

    
def cluster_bootstrap_cis(y, p, g, B=5000, alpha=0.05, thr=0.5, seed=0):
    """
    Computes confidence intervals for performance metrics
    Uses cluster bootstrap resampling
    y: array of outcomes
    p: array of predicted probabilites
    g: array of group IDs (e.g. patient IDs)
    """
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    p = np.asarray(p)
    g = np.asarray(g)

    # Fing unique group IDs and corresponding indices
    uniq = np.unique(g)
    idx_by_g = {gg: np.where(g == gg)[0] for gg in uniq}

    point = compute_metrics(y, p, thr=thr) # point estimates
    boot = {k: [] for k in point.keys()}

    for i in range(B):
        sampled = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([idx_by_g[gg] for gg in sampled])
        m = compute_metrics(y[idx], p[idx], thr=thr)
        for k, v in m.items():
            boot[k].append(v)

        if (i + 1) % 500 == 0:
            print(f"{i + 1} resamples complete")

        
    out = {}
    for k, v in boot.items():
        vals = np.asarray(v, dtype=float)
        lo, hi = np.nanquantile(v, [alpha/2, 1 - alpha/2])
        out[k] = (point[k], lo, hi)
    return out

## Calculation of Wilson CIs
def wilson_ci(k, n, alpha=0.05):
    """
    Wilson confidence interval for a binomial proportion.
    
    Parameters
    k : int or float
        Number of events.
    n : int
        Number of observations.
    alpha : float
        Significance level. i.e. alpha=0.05 gives a 95% CI.

    Returns
    lower, upper : float (lower and upper confidence bounds)
    """
    if n == 0:
        return np.nan, np.nan

    z = norm.ppf(1 - alpha / 2)
    p = k / n

    denominator = 1 + z**2 / n
    centre = p + z**2 / (2 * n)
    half_width = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n)

    lower = (centre - half_width) / denominator
    upper = (centre + half_width) / denominator

    return lower, upper

#----------------------------------------------------------------------------------
# Custom pre-processers

## Implementation of capping at selected percentiles

class Capper(BaseEstimator, TransformerMixin):
    """
    This transformer implements capping at the specified percentiles
    Capping replaces extreme values with the value at the chosen percentile
    
    Example use:
    #num_pipe = Pipeline([
    ("cap", Capper(1, 99)),
    ("scale", StandardScaler())
    ])
    """
    def __init__(self, low=1, high=99):
        self.low = low
        self.high = high

    def fit(self, X, y=None):
        self.lower_ = np.percentile(X, self.low, axis=0)
        self.upper_ = np.percentile(X, self.high, axis=0)
        return self

    def transform(self, X):
        return np.clip(X, self.lower_, self.upper_)

    
    def get_feature_names_out(self, input_features=None):
        return input_features

## Data type conversion
def to_object_dtype(df):
    return df.astype("object")

to_object = FunctionTransformer(
    func=to_object_dtype,
    feature_names_out="one-to-one"
)

In [ ]:
# MLP with a wrapper to enable Bayesian tuning of hidden layer sizes

class TunableMLPClassifier(ClassifierMixin, BaseEstimator):
    def __init__(
        self,
        n_hidden_layers=1,
        hidden_layer_1=20,
        hidden_layer_2=20,
        activation="relu",
        solver="adam",
        alpha=0.0001,
        batch_size="auto",
        learning_rate="constant",
        learning_rate_init=0.001,
        power_t=0.5,
        max_iter=1000,
        shuffle=True,
        random_state=None,
        tol=0.0001,
        verbose=False,
        warm_start=False,
        momentum=0.9,
        nesterovs_momentum=True,
        early_stopping=False,
        validation_fraction=0.1,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-8,
        n_iter_no_change=10,
        max_fun=15000,
    ):
        self.n_hidden_layers = n_hidden_layers
        self.hidden_layer_1 = hidden_layer_1
        self.hidden_layer_2 = hidden_layer_2

        self.activation = activation
        self.solver = solver
        self.alpha = alpha
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.learning_rate_init = learning_rate_init
        self.power_t = power_t
        self.max_iter = max_iter
        self.shuffle = shuffle
        self.random_state = random_state
        self.tol = tol
        self.verbose = verbose
        self.warm_start = warm_start
        self.momentum = momentum
        self.nesterovs_momentum = nesterovs_momentum
        self.early_stopping = early_stopping
        self.validation_fraction = validation_fraction
        self.beta_1 = beta_1
        self.beta_2 = beta_2
        self.epsilon = epsilon
        self.n_iter_no_change = n_iter_no_change
        self.max_fun = max_fun

    def _get_hidden_layer_sizes(self):
        if self.n_hidden_layers == 1:
            return (self.hidden_layer_1,)
        elif self.n_hidden_layers == 2:
            return (self.hidden_layer_1, self.hidden_layer_2)
        else:
            raise ValueError(
                "n_hidden_layers must be either 1 or 2. "
                f"Got {self.n_hidden_layers}."
            )

    def fit(self, X, y):
        hidden_layer_sizes = self._get_hidden_layer_sizes()

        self.model_ = MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            activation=self.activation,
            solver=self.solver,
            alpha=self.alpha,
            batch_size=self.batch_size,
            learning_rate=self.learning_rate,
            learning_rate_init=self.learning_rate_init,
            power_t=self.power_t,
            max_iter=self.max_iter,
            shuffle=self.shuffle,
            random_state=self.random_state,
            tol=self.tol,
            verbose=self.verbose,
            warm_start=self.warm_start,
            momentum=self.momentum,
            nesterovs_momentum=self.nesterovs_momentum,
            early_stopping=self.early_stopping,
            validation_fraction=self.validation_fraction,
            beta_1=self.beta_1,
            beta_2=self.beta_2,
            epsilon=self.epsilon,
            n_iter_no_change=self.n_iter_no_change,
            max_fun=self.max_fun,
        )

        self.model_.fit(X, y)

        self.classes_ = self.model_.classes_
        self.n_features_in_ = self.model_.n_features_in_

        if hasattr(self.model_, "feature_names_in_"):
            self.feature_names_in_ = self.model_.feature_names_in_

        return self

    def predict(self, X):
        check_is_fitted(self, "model_")
        return self.model_.predict(X)

    def predict_proba(self, X):
        check_is_fitted(self, "model_")
        return self.model_.predict_proba(X)

    def predict_log_proba(self, X):
        check_is_fitted(self, "model_")
        return self.model_.predict_log_proba(X)

    @property
    def hidden_layer_sizes_(self):
        check_is_fitted(self, "model_")
        return self.model_.hidden_layer_sizes

In [ ]:
# Feature dictionary
## Used for presentation of results

feature_dict = {
    "albumin_latest": "Latest serum albumin",
    "bilirubin_max": "Highest serum bilirubin",
    "pH_min": "Lowest pH",
    "lactate_max": "Highest lactate",
    "ethnicity_grouped": "Ethnicity",
    "latest_temperature": "Latest body temperature",
    "sodium_min": "Lowest serum sodium",
    "Hb_latest": "Latest haemoglobin",
    "plt_latest": "Latest platelet count",
    "plt_min": "Lowest platelet count",
    "potassium_max": "Highest serum potassium",
    "potassium_min": "Lowest serum potassium",
    "sodium_max": "Highest serum sodium",
    "creatinine_max": "Highest serum creatinine",
    "WCC_max": "Highest white cell count",
    "WCC_latest": "Latest white cell count",
    "urea_nitrogen_max": "Highest urea nitrogen",
    "Charlson_count": "Charlson condition count",
    "sex": "Sex",
    "insurance": "Health insurance",
    "admissions_in_past_year": "Admissions in preceding year",
    "hospital_days_past_year": "Inpatient days in preceding year",
    "discharge_dest": "Discharge destination",
    "admission_type": "Admission type",
    "hosp_los": "Hospital length of stay",
    "tracheostomy": "Tracheostomy performed",
    "rbc_transfusion": "Red blood cell transfusion",
    "age_at_discharge": "Age at hospital discharge",
    "mv_days": "Days of mechanical ventilation",
    "vaso_days": "Days of vasoactive support",
    "max_na_eq_dose": "Maximum vasopressor dose",
    "rrt_calendar_days": "Days of RRT",
    "n_of_icu_admissions": "Number of ICU admissions",
    "DNACPR": "Resuscitation status",
    "latest_sbp": "Latest systolic blood pressure",
    "lowest_gcs": "Lowest GCS",
    "discharge_dest_NOT HOME": "Discharge destination: not home",
    "discharge_dest_HOME HEALTH CARE": "Discharge destination: home healthcare",
    "discharge_dest_HOME": "Discharge destination: home",     
    "ethnicity_grouped_white": "Ethnicity: white",
    "ethnicity_grouped_not black white or hispanic": "Ethnicity: not black, white or hispanic",
    "ethnicity_grouped_hispanic": "Ethnicity: hispanic",
    "ethnicity_grouped_black": "Ethnicity: black",
    "insurance_Private": "Health insurance: private",
    "insurance_Medicaid": "Health insurance: Medicaid",
    "insurance_Medicare": "Health insurance: Medicare",
    "insurance_Other": "Health insurance: other",
    "sex_M": "Sex: male",
    "sex_F": "Sex: female",
    "admission_type_emergency_surgical": "Admission type: emergency surgical",
    "admission_type_emergency_non-surgical": "Admission type: emergency non-surgical",
    "admission_type_elective": "Admission type: elective"
}

In [ ]:
# Set file paths
#--------------------------------------------------------------------------------------------------
## Training set

X = pd.read_parquet(r"")
y = pd.read_pickle(r"")

#--------------------------------------------------------------------------------------------------
## Held-out test set

X_test = pd.read_parquet(r"")
y_test = pd.read_pickle(r"")

#--------------------------------------------------------------------------------------------------
## Prefix for results filenames

PREFIX = "MLP_30d_readmission"

#--------------------------------------------------------------------------------------------------
## Reference categories for one-hot encoding

ref_cats = ["HOME", 
            #"F", 
            "not black white or hispanic", 
            "elective", 
            "Other", 
            0, 
            #0, 
            #0
           ]

### Train the model using nested K-fold cross-validation

In [ ]:
start_time = time.time()

# Set seed sequence for reproducibility
master_seed = 7356
ss = np.random.SeedSequence(master_seed)
SEED_OUTERSPLIT, SEED_INNERSPLIT, SEED_IMPUTE, SEED_BAYES, SEED_MODEL = [int(s.generate_state(1)[0]) for s in ss.spawn(5)]

#--------------------------------------------------------------------------------------------------------------
## Step 1) Define groups and column types
## NB. for the MLP, we use the predictors that were selected by the logistic regression model

groups=X.loc[X.index, "subject_id"].to_numpy()
subject_id = X["subject_id"].values

num_cols = [
    'Charlson_count', 'WCC_latest', #'age_at_discharge', 
    'admissions_in_past_year', 'hosp_los', #'plt_min',  
    'bilirubin_max', #'latest_temperature', 'pH_min',
    'Hb_latest', 'albumin_latest', 'plt_latest', #'WCC_max',
    'urea_nitrogen_max', #'creatinine_max',
    # 'lactate_max',  
    #'potassium_max', 'potassium_min', 'sodium_max', 'sodium_min',
    #  'latest_sbp',
    # 'lowest_gcs', 'max_na_eq_dose', 'hospital_days_past_year',
    #'n_of_icu_admissions',  
    #'rrt_calendar_days', 'mv_days', 'vaso_days', 
]
              
X[num_cols] = X[num_cols].astype("float64") # Convert pd.NA to np.nan

cat_cols = [
    'discharge_dest', 'ethnicity_grouped', #'sex',
    'admission_type', 'insurance',
    'DNACPR', #'tracheostomy', 'rbc_transfusion'
]

#-------------------------------------------------------------------------------------------------------------
# Step 2) Apply the pre-processing required to support imputation
#  - One-hot encode categorical variables (dense, to allow IterativeImputer to function)

pre_impute = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat",
         Pipeline(steps=[
             ("to_object", to_object),
             ("onehot", OneHotEncoder(
                 handle_unknown="ignore",
                 drop=ref_cats,
                 sparse_output=False   # Required for IterativeImputer
             )),
         ]),
         cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

n_num = len(num_cols)  # To identify numerical columns later 

#-------------------------------------------------------------------------------------------------------------
# Step 3) Deterministic regression imputation of missing values.  All predictor variables
# are included in the imputation model

imputer = IterativeImputer(
    estimator=BayesianRidge(),
    sample_posterior=False, # Deterministic regression imputation
    max_iter=10,
    random_state=SEED_IMPUTE,
    min_value=0
)

#-------------------------------------------------------------------------------------------------------------
# Step 4) Post-imputation steps on first n_num columns (numerical columns only)
# - cap at 1st and 99th centiles
# - standardise
# NB One-hot encoded columns pass through unchanged.
# -----------------------------
numerical_post = Pipeline(steps=[
    ("cap", Capper(1, 99)),
    ("scaler", StandardScaler(with_mean=True)),
])

postprocess = ColumnTransformer(
    transformers=[
        ("num_post", numerical_post, slice(0, n_num)),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

#-------------------------------------------------------------------------------------------------------------
# Step 5) Classifier (MLP with 1-2 hidden layers)
clf = TunableMLPClassifier(
    activation="relu",
    solver="adam",
    batch_size="auto",
    learning_rate_init=0.001,
    alpha=0.0001,
    max_iter=1000,
    random_state=SEED_MODEL
)

#-------------------------------------------------------------------------------------------------------------
# Step 6) Recursive feature elimination: not applicable

#-------------------------------------------------------------------------------------------------------------
# Step 7) Assemble pipeline
pipe = Pipeline(steps=[
    ("pre_impute", pre_impute),
    ("imputer", imputer),
    ("postprocess", postprocess),
    ("clf", clf),
])

print(pipe.named_steps["pre_impute"].transformers)

#-------------------------------------------------------------------------------------------------------------
# Step 8) Define hyperparameter grid 

search_spaces = {
    "clf__n_hidden_layers": Categorical([1, 2]),
    "clf__hidden_layer_1": Integer(5, 40),
    "clf__hidden_layer_2": Integer(5, 40),
    "clf__alpha": Real(1e-6, 1e-1, prior="log-uniform"),
    "clf__learning_rate_init": Real(1e-5, 1e-1, prior="log-uniform")
}


#-------------------------------------------------------------------------------------------------------------
# Step 9) Set up and run the nested K-fold cross-validation

inner_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED_INNERSPLIT)
outer_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED_OUTERSPLIT)

#......................................................................................
# A. Inner cross-validation for hyperparameter tuning
gs = BayesSearchCV(
    estimator=pipe,
    search_spaces=search_spaces,
    n_iter=50,       
    scoring="roc_auc",
    cv=inner_cv,
    n_jobs=1,
    n_points=5,           
    random_state=SEED_BAYES,
    verbose=1,
)

scoring = {"AUROC": "roc_auc", "Brier score": "neg_brier_score",
           "Negative log loss": "neg_log_loss", "Accuracy": "accuracy",
           "Precision": "precision", "Recall": "recall",
           "F1 score": "f1"}

#......................................................................................
# B. Outer cross-validation for performance evaluation
out = cross_validate(
    estimator=gs,
    X=X,
    y=y,
    groups=groups, # grouped splitting for outer folds
    params={"groups": groups}, # grouped splitting passed to inner folds
    cv=outer_cv,
    scoring=scoring,
    n_jobs=-1,
    return_estimator=True,
    return_indices=True,
    verbose=1,
    error_score="raise"
)

end_time = time.time()
print(f"Total training runtime is {(end_time - start_time)/60} minutes")

In [ ]:
# Inspect the chosen hyperparameters
best_params = [est.best_params_ for est in out["estimator"]]
best_params

### Model evaluation: outer K-fold cross-validation

In [ ]:
# Evaluate the model
# -------------------------------------------------------------------------------------------
# Step 1) Recover the predicted probabilities, to enable generation of calibration curves etc.
start_time = time.time()

n = len(y)
oof_proba = np.full(n, np.nan, dtype=float) # out of fold y_pred

for est, test_idx in zip(out["estimator"], out["indices"]["test"]):
    X_te = X.iloc[test_idx] 
    oof_proba[test_idx] = est.predict_proba(X_te)[:, 1]

np.save(f"{PREFIX}_oof_y_pred.npy", oof_proba)

# -------------------------------------------------------------------------------------------
# Step 2) Compute calibration metrics (including calibration curve)

# A. Simple metrics
oe = observed_expected_ratio(y, oof_proba)
calib_metrics = calibration_intercept_slope(
    y,
    oof_proba,
    epsilon=1e-6,
    max_iter=1000,
    alpha=0.05
)

#..........................................................
# B. Flexible smoothed calibration curve with 95% CI bands
## Set the number of bootstrap resamples
B = 500

y_true = np.asarray(y).astype(float)
p = np.clip(np.asarray(oof_proba).astype(float), 1e-6, 1 - 1e-6) # To avoid 0s or 1s

## Fit the smoothed calibration curve
grid = np.linspace(1e-6, 1 - 1e-6, 200)
fit = sm.nonparametric.lowess(endog=y_true,
                              exog=p, 
                              xvals=grid, 
                              frac=0.25, 
                              it=0)

#............................................................
## Implement the bootstrap to get confidence bands
## NB the bootstrap is "clustered" by subject_ID, since rows from the same patient are not independent
rng = np.random.default_rng()
subject_id = np.asarray(subject_id)
groups, inv = np.unique(subject_id, return_inverse=True) # Assign groups based on subject_id
G = len(groups)
group_rows = [np.flatnonzero(inv == g) for g in range(G)] # Compute row indices for each group 

boots = np.empty((B, len(grid)), dtype=float) # Empty grid to fill with resamples

for b in range(B):
    sampled_g = rng.integers(0, G, size=G)       # sample G groups (with replacement)
    idx = np.concatenate([group_rows[g] for g in sampled_g])

    ## Fit LOWESS in each bootstrap resample
    boots[b, :] = sm.nonparametric.lowess(
        endog=y_true[idx],
        exog=p[idx],
        xvals=grid,
        frac=0.25,
        it=0,
    )

   ## Monitor progress 
    if (b + 1) % 100 == 0:
        print(f"{b + 1} resamples complete")


## Return low and high confidence bands
lo = np.quantile(boots, 0.025, axis=0)
hi = np.quantile(boots, 0.975, axis=0)

In [ ]:
# Plot curves from K-fold cross-validation
plt.style.use('ggplot')

# Plot the smoothed calibration curve with confidence bands and rug plot
fig, ax = plt.subplots(figsize=(6, 5))

ax.grid(False)
ax.set_facecolor("white")

for spine in ["bottom", "left"]:
    ax.spines[spine].set_color("black")
    ax.spines[spine].set_linewidth(1)

ax.plot([0, 1], [0, 1], "--", color="grey", linewidth=1, label="Ideal")
ax.plot(grid, fit, color="navy", linewidth=1.3, label="Smoothed calibration curve")
ax.fill_between(grid, lo, hi, color="grey", alpha=0.33, label="95% confidence interval")

ax.set_xlabel("Predicted probability", fontsize=10)
ax.set_ylabel("Observed probability", fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xticks([0.2, 0.4, 0.6, 0.8, 1])
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1])
ax.tick_params(axis="both", labelsize=8)

# rug plot parameters
bins = 100 # adjustable
counts, edges = np.histogram(oof_proba, bins=bins, range=(0, 1)) # creates two arrays
centrepoints = 0.5 * (edges[:-1] + edges[1:])
rug_frac = 0.15 # scale relative to y axis
heights = (counts / counts.max()) * rug_frac 

ax.vlines(
    centrepoints, 0, heights,
    transform=ax.get_xaxis_transform(),  # x in data coordinates, y in axis coordinates
    color="black", alpha=0.3, linewidth=1,
    clip_on=False
)

ax.legend(fontsize=8)
#plt.show()
plt.savefig(f"{PREFIX}_outoffold_calibration_curve.tiff", bbox_inches="tight", dpi=600)

#--------------------------------------------------------------------------------------

# Plot the quantile calibration plot
n_bins = 10
alpha = 0.05
df = pd.DataFrame({
    "y_true": np.asarray(y),
    "pred_prob": np.asarray(oof_proba)
})
n_unique_pred_probs = df["pred_prob"].nunique()

if n_unique_pred_probs <= n_bins:
    # For models with few probability outputs, group directly by the exact predicted probability.
    df["prob_bin"] = df["pred_prob"]

else:
    # Otherwise use deciles of predicted probability.
    df["prob_bin"] = pd.qcut(
        df["pred_prob"],
        q=n_bins,
        duplicates="drop"
    )

## Summarise calibration per bin
calibration = (
    df
    .groupby("prob_bin", observed=True)
    .agg(
        n=("y_true", "size"),
        events=("y_true", "sum"),
        mean_pred_prob=("pred_prob", "mean"),
        min_pred_prob=("pred_prob", "min"),
        max_pred_prob=("pred_prob", "max")
    )
    .reset_index()
)
calibration["observed_prob"] = calibration["events"] / calibration["n"]

## Calculate Wilson confidence intervals for observed probabilities
ci_values = [
    wilson_ci(k=row["events"], n=row["n"], alpha=alpha)
    for _, row in calibration.iterrows()
]

calibration["ci_lower"] = [ci[0] for ci in ci_values]
calibration["ci_upper"] = [ci[1] for ci in ci_values]

calibration = calibration.sort_values("mean_pred_prob").reset_index(drop=True) # Sort predicted probabilities

## Plot
fig, ax = plt.subplots(figsize=(6, 5))

ax.grid(False)
ax.set_facecolor("white")

for spine in ["bottom", "left"]:
    ax.spines[spine].set_color("black")
    ax.spines[spine].set_linewidth(1)

x_plot = calibration["mean_pred_prob"].to_numpy()
y_plot = calibration["observed_prob"].to_numpy()

yerr_lower = y_plot - calibration["ci_lower"].to_numpy()
yerr_upper = calibration["ci_upper"].to_numpy() - y_plot

ax.errorbar(
    x_plot,
    y_plot,
    yerr=np.vstack([yerr_lower, yerr_upper]),
    fmt="o",
    capsize=3,
    markersize=5,
    color="darkolivegreen",
    linestyle="none"
)

ax.plot([0, 1], [0, 1], "--", color="grey", linewidth=1, label="Ideal calibration")
ax.set_xlabel("Mean predicted probability", fontsize=10)
ax.set_ylabel("Observed probability", fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xticks([0.2, 0.4, 0.6, 0.8, 1])
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1])
ax.tick_params(axis="both", labelsize=8)

#plt.show()
plt.savefig(f"{PREFIX}_outoffold_decile_calibration_plot.tiff", bbox_inches="tight", dpi=600)

#--------------------------------------------------------------------------------------

# Plot the ROC curve
fpr, tpr, _ = roc_curve(y, oof_proba)  
auc = roc_auc_score(y, oof_proba)      

fig, ax = plt.subplots(figsize=(6, 5))

ax.grid(False)
ax.set_facecolor("white")

for spine in ["bottom", "left"]:
    ax.spines[spine].set_color("black")
    ax.spines[spine].set_linewidth(1)
    
ax.plot(fpr, tpr, linewidth=1.3, color="darkred", label=f"AUC={auc:.3f}")
ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1, label="Chance")
ax.set_xlabel("False positive rate", fontsize=10)
ax.set_ylabel("True positive rate", fontsize=10)
ax.set_xlim(0, 1)
ax.set_xticks([0.2, 0.4, 0.6, 0.8, 1])
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1])
ax.set_ylim(0, 1)
ax.tick_params(axis="both", labelsize=8)
ax.annotate(
    f"AUC: {auc:.3f}",
    xy=(0.05, 0.9), 
    fontsize=10,
    alpha=0.75
)
#plt.show()
plt.savefig(f"{PREFIX}_outoffold_roc_curve.tiff", bbox_inches="tight", dpi=600)
#--------------------------------------------------------------------------------------

# Plot precision recall curve
precision, recall, thresholds = precision_recall_curve(y, oof_proba)
auprc = average_precision_score(y, oof_proba)

fig, ax = plt.subplots(figsize=(6, 5))

ax.grid(False)
ax.set_facecolor("white")
for spine in ["bottom", "left"]:
    ax.spines[spine].set_color("black")
    ax.spines[spine].set_linewidth(1)

ax.plot(
    recall,
    precision,
    color="#B45309",
    linewidth=1.1,
)

ax.set_xlabel("Recall (sensitivity)", fontsize=10)
ax.set_ylabel("Precision (PPV)", fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xticks([0.2, 0.4, 0.6, 0.8, 1])
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1])
ax.tick_params(axis="both", labelsize=8)
ax.annotate(
    f"AUPRC: {auprc:.3f}",
    xy=(0.05, 0.5), 
    fontsize=10,
    alpha=0.75
)

#plt.show()
plt.savefig(f"{PREFIX}_outoffold_pr_curve.tiff", bbox_inches="tight", dpi=600)
#--------------------------------------------------------------------------------------

end_time = time.time()
print(f"Total evaluation runtime is {(end_time - start_time)/60} minutes")

In [ ]:
# Calculate and collate performance metrics

result = cluster_bootstrap_cis(y, oof_proba, subject_id)
result

df_ci = (
    pd.DataFrame.from_dict(result, orient="index", columns=["score", "ci_low", "ci_high"])
      .reset_index()
      .rename(columns={"index": "metric"})
)

oe = oe_ratio_ci_cluster_bootstrap(y, oof_proba, subject_id)

cal = {**calib_metrics, **oe}

rows = []
for k, v in cal.items():
    if k.endswith("_ci_95"):
        continue
    ci_low, ci_high = cal.get(f"{k}_ci_95", (np.nan, np.nan))
    rows.append({"metric": k, "score": v, "ci_low": ci_low, "ci_high": ci_high})


cal_df = pd.DataFrame(rows, columns=["metric", "score", "ci_low", "ci_high"])

metrics_df = pd.concat([df_ci, cal_df], ignore_index=True)

metrics_df.to_csv(f"{PREFIX}_outoffold_metrics.csv")

metrics_df

### Re-train the model on the entire training set

In [ ]:
# Fit the final model using the whole training dataset
start_time = time.time()

gs = BayesSearchCV(
    estimator=pipe,
    search_spaces=search_spaces,
    n_iter=50,               
    scoring="roc_auc",
    cv=inner_cv,
    n_jobs=-1, 
    n_points=5,           
    random_state=SEED_BAYES,
    verbose=1,
)

# Fit the model
gs.fit(X, y, groups=subject_id)

end_time = time.time()
print(f"Total runtime is {(end_time - start_time)/60} minutes")

In [ ]:
# Access model attributes

final_model = gs.best_estimator_
final_params = gs.best_params_
clf = final_model.named_steps["clf"]

# Model steps
pre = final_model.named_steps["pre_impute"]
post = final_model.named_steps["postprocess"]

# Access names after pre_impute stage
names_pre = pre.get_feature_names_out()
names_post = post.get_feature_names_out(names_pre)

# Get features
names_selected = np.asarray(names_post)

In [ ]:
# Save the model
joblib.dump(final_model, f"{PREFIX}_model.pkl")
#final_model = joblib.load(f"{PREFIX}_model.pkl)

### Evaluate apparent model performance
- Apply the fitted model to the whole training set
- Calculate performance metrics

In [ ]:
# Evaluate the model
# -------------------------------------------------------------------------------------------
# Step 1) Compute predicted probabilities, to enable generation of calibration curves etc.
start_time = time.time()

n = len(y)
apparent_y_pred = final_model.predict_proba(X)[:, 1]

np.save(f"{PREFIX}_apparent_y_pred.npy", apparent_y_pred)

# A. Simple metrics
oe = observed_expected_ratio(y, apparent_y_pred)
calib_metrics = calibration_intercept_slope(
    y,
    apparent_y_pred,
    epsilon=1e-6,
    max_iter=1000,
    alpha=0.05
)

#..........................................................
# B. Flexible smoothed calibration curve with 95% CI bands
## Set the number of bootstrap resamples
B = 500

y_true = np.asarray(y).astype(float)
p = np.clip(np.asarray(apparent_y_pred).astype(float), 1e-6, 1 - 1e-6) # To avoid 0s or 1s

## Fit the smoothed calibration curve
grid = np.linspace(1e-6, 1 - 1e-6, 200)
fit = sm.nonparametric.lowess(endog=y_true,
                              exog=p, 
                              xvals=grid, 
                              frac=0.25, 
                              it=0)

#............................................................
## Implement the bootstrap to get confidence bands
## NB the bootstrap is "clustered" by subject_ID, since rows from the same patient are not independent
rng = np.random.default_rng()
subject_id = np.asarray(subject_id)
groups, inv = np.unique(subject_id, return_inverse=True) # Assign groups based on subject_id
G = len(groups)
group_rows = [np.flatnonzero(inv == g) for g in range(G)] # Compute row indices for each group 

boots = np.empty((B, len(grid)), dtype=float) # Empty grid to fill with resamples

for b in range(B):
    sampled_g = rng.integers(0, G, size=G)       # sample G groups (with replacement)
    idx = np.concatenate([group_rows[g] for g in sampled_g])

    ## Fit LOWESS in each bootstrap resample
    boots[b, :] = sm.nonparametric.lowess(
        endog=y_true[idx],
        exog=p[idx],
        xvals=grid,
        frac=0.25,
        it=0,
    )

   ## Monitor progress 
    if (b + 1) % 100 == 0:
        print(f"{b + 1} resamples complete")


## Return low and high confidence bands
lo = np.quantile(boots, 0.025, axis=0)
hi = np.quantile(boots, 0.975, axis=0)

In [ ]:
# Plot curves for apparent performance
plt.style.use('ggplot')

# Plot the smoothed calibration curve with confidence bands and rug plot
fig, ax = plt.subplots(figsize=(6, 5))

ax.grid(False)
ax.set_facecolor("white")

for spine in ["bottom", "left"]:
    ax.spines[spine].set_color("black")
    ax.spines[spine].set_linewidth(1)

ax.plot([0, 1], [0, 1], "--", color="grey", linewidth=1, label="Ideal")
ax.plot(grid, fit, color="navy", linewidth=1.3, label="Smoothed calibration curve")
ax.fill_between(grid, lo, hi, color="grey", alpha=0.33, label="95% confidence interval")

ax.set_xlabel("Predicted probability", fontsize=10)
ax.set_ylabel("Observed probability", fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xticks([0.2, 0.4, 0.6, 0.8, 1])
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1])
ax.tick_params(axis="both", labelsize=8)

# rug plot parameters
bins = 100 # adjustable
counts, edges = np.histogram(apparent_y_pred, bins=bins, range=(0, 1)) # creates two arrays
centrepoints = 0.5 * (edges[:-1] + edges[1:])
rug_frac = 0.15 # scale relative to y axis
heights = (counts / counts.max()) * rug_frac 

ax.vlines(
    centrepoints, 0, heights,
    transform=ax.get_xaxis_transform(),  # x in data coordinates, y in axis coordinates
    color="black", alpha=0.3, linewidth=1,
    clip_on=False
)

ax.legend(fontsize=8)
#plt.show()
plt.savefig(f"{PREFIX}_apparent_calibration_curve.tiff", bbox_inches="tight", dpi=600)

#--------------------------------------------------------------------------------------

# Plot the quantile calibration plot
n_bins = 10
alpha = 0.05
df = pd.DataFrame({
    "y_true": np.asarray(y),
    "pred_prob": np.asarray(apparent_y_pred)
})
n_unique_pred_probs = df["pred_prob"].nunique()

if n_unique_pred_probs <= n_bins:
    # For models with few probability outputs, group directly by the exact predicted probability.
    df["prob_bin"] = df["pred_prob"]

else:
    # Otherwise use deciles of predicted probability.
    df["prob_bin"] = pd.qcut(
        df["pred_prob"],
        q=n_bins,
        duplicates="drop"
    )

## Summarise calibration per bin
calibration = (
    df
    .groupby("prob_bin", observed=True)
    .agg(
        n=("y_true", "size"),
        events=("y_true", "sum"),
        mean_pred_prob=("pred_prob", "mean"),
        min_pred_prob=("pred_prob", "min"),
        max_pred_prob=("pred_prob", "max")
    )
    .reset_index()
)
calibration["observed_prob"] = calibration["events"] / calibration["n"]

## Calculate Wilson confidence intervals for observed probabilities
ci_values = [
    wilson_ci(k=row["events"], n=row["n"], alpha=alpha)
    for _, row in calibration.iterrows()
]

calibration["ci_lower"] = [ci[0] for ci in ci_values]
calibration["ci_upper"] = [ci[1] for ci in ci_values]

calibration = calibration.sort_values("mean_pred_prob").reset_index(drop=True) # Sort predicted probabilities

## Plot
fig, ax = plt.subplots(figsize=(6, 5))

ax.grid(False)
ax.set_facecolor("white")

for spine in ["bottom", "left"]:
    ax.spines[spine].set_color("black")
    ax.spines[spine].set_linewidth(1)

x_plot = calibration["mean_pred_prob"].to_numpy()
y_plot = calibration["observed_prob"].to_numpy()

yerr_lower = y_plot - calibration["ci_lower"].to_numpy()
yerr_upper = calibration["ci_upper"].to_numpy() - y_plot

ax.errorbar(
    x_plot,
    y_plot,
    yerr=np.vstack([yerr_lower, yerr_upper]),
    fmt="o",
    capsize=3,
    markersize=5,
    color="darkolivegreen",
    linestyle="none"
)

ax.plot([0, 1], [0, 1], "--", color="grey", linewidth=1, label="Ideal calibration")
ax.set_xlabel("Mean predicted probability", fontsize=10)
ax.set_ylabel("Observed probability", fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xticks([0.2, 0.4, 0.6, 0.8, 1])
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1])
ax.tick_params(axis="both", labelsize=8)

#plt.show()
plt.savefig(f"{PREFIX}_apparent_decile_calibration_plot.tiff", bbox_inches="tight", dpi=600)


#--------------------------------------------------------------------------------------

# Plot the ROC curve
fpr, tpr, _ = roc_curve(y, apparent_y_pred)  
auc = roc_auc_score(y, apparent_y_pred)      

fig, ax = plt.subplots(figsize=(6, 5))

ax.grid(False)
ax.set_facecolor("white")

for spine in ["bottom", "left"]:
    ax.spines[spine].set_color("black")
    ax.spines[spine].set_linewidth(1)
    
ax.plot(fpr, tpr, linewidth=1.3, color="darkred", label=f"AUC={auc:.3f}")
ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1, label="Chance")
ax.set_xlabel("False positive rate", fontsize=10)
ax.set_ylabel("True positive rate", fontsize=10)
ax.set_xlim(0, 1)
ax.set_xticks([0.2, 0.4, 0.6, 0.8, 1])
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1])
ax.set_ylim(0, 1)
ax.tick_params(axis="both", labelsize=8)
ax.annotate(
    f"AUC: {auc:.3f}",
    xy=(0.05, 0.9), 
    fontsize=10,
    alpha=0.75
)
#plt.show()
plt.savefig(f"{PREFIX}_apparent_roc_curve.tiff", bbox_inches="tight", dpi=600)
#--------------------------------------------------------------------------------------

# Plot precision recall curve
precision, recall, thresholds = precision_recall_curve(y, apparent_y_pred)
auprc = average_precision_score(y, apparent_y_pred)

fig, ax = plt.subplots(figsize=(6, 5))

ax.grid(False)
ax.set_facecolor("white")
for spine in ["bottom", "left"]:
    ax.spines[spine].set_color("black")
    ax.spines[spine].set_linewidth(1)

ax.plot(
    recall,
    precision,
    color="#B45309",
    linewidth=1.1,
)

ax.set_xlabel("Recall (sensitivity)", fontsize=10)
ax.set_ylabel("Precision (PPV)", fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xticks([0.2, 0.4, 0.6, 0.8, 1])
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1])
ax.tick_params(axis="both", labelsize=8)
ax.annotate(
    f"AUPRC: {auprc:.3f}",
    xy=(0.05, 0.5), 
    fontsize=10,
    alpha=0.75
)

#plt.show()
plt.savefig(f"{PREFIX}_apparent_pr_curve.tiff", bbox_inches="tight", dpi=600)

In [ ]:
result = cluster_bootstrap_cis(y, apparent_y_pred, subject_id)
result

df_ci = (
    pd.DataFrame.from_dict(result, orient="index", columns=["score", "ci_low", "ci_high"])
      .reset_index()
      .rename(columns={"index": "metric"})
)

oe = oe_ratio_ci_cluster_bootstrap(y, apparent_y_pred, subject_id)

cal = {**calib_metrics, **oe}

rows = []
for k, v in cal.items():
    if k.endswith("_ci_95"):
        continue
    ci_low, ci_high = cal.get(f"{k}_ci_95", (np.nan, np.nan))
    rows.append({"metric": k, "score": v, "ci_low": ci_low, "ci_high": ci_high})


cal_df = pd.DataFrame(rows, columns=["metric", "score", "ci_low", "ci_high"])

metrics_df = pd.concat([df_ci, cal_df], ignore_index=True)

metrics_df.to_csv(f"{PREFIX}_apparent_metrics.csv")

metrics_df

### Evaluate model in held-out test set

In [ ]:
# Apply same column groups/settings as previously
groups=X_test.loc[X_test.index, "subject_id"].to_numpy()
subject_id = X_test["subject_id"].values

X_test[num_cols] = X_test[num_cols].astype("float64") # Convert pd.NA to np.nan

# Make predictions
y_probas = final_model.predict_proba(X_test)

y_prob = y_probas[:,1] # positive class predictions

np.save(f"{PREFIX}_testset_y_pred.npy", y_prob)

In [ ]:
# Evaluate the model
# -------------------------------------------------------------------------------------------
# Step 1) Compute calibration metrics (including calibration curve)
# A. Simple metrics
oe = observed_expected_ratio(y_test, y_prob)
calib_metrics = calibration_intercept_slope(
    y_test,
    y_prob,
    epsilon=1e-6,
    max_iter=1000,
    alpha=0.05
)

#..........................................................
# B. Flexible smoothed calibration curve with 95% CI bands
## Set the number of bootstrap resamples
B = 500

y_true = np.asarray(y_test).astype(float)
p = np.clip(np.asarray(y_prob).astype(float), 1e-6, 1 - 1e-6) # To avoid 0s or 1s

## Fit the smoothed calibration curve
grid = np.linspace(1e-6, 1 - 1e-6, 200)
fit = sm.nonparametric.lowess(endog=y_test,
                              exog=y_prob, 
                              xvals=grid, 
                              frac=0.25, 
                              it=0)

#............................................................
## Implement the bootstrap to get confidence bands
## NB the bootstrap is "clustered" by subject_ID, since rows from the same patient are not independent
rng = np.random.default_rng()
subject_id = np.asarray(subject_id)
groups, inv = np.unique(subject_id, return_inverse=True) # Assign groups based on subject_id
G = len(groups)
group_rows = [np.flatnonzero(inv == g) for g in range(G)] # Compute row indices for each group 

boots = np.empty((B, len(grid)), dtype=float) # Empty grid to fill with resamples

for b in range(B):
    sampled_g = rng.integers(0, G, size=G)       # sample G groups (with replacement)
    idx = np.concatenate([group_rows[g] for g in sampled_g])

    ## Fit LOWESS in each bootstrap resample
    boots[b, :] = sm.nonparametric.lowess(
        endog=y_true[idx],
        exog=p[idx],
        xvals=grid,
        frac=0.25,
        it=0,
    )

   ## Monitor progress 
    if (b + 1) % 100 == 0:
        print(f"{b + 1} resamples complete")


## Return low and high confidence bands
lo = np.quantile(boots, 0.025, axis=0)
hi = np.quantile(boots, 0.975, axis=0)

In [ ]:
# Plot curves from split-sample validation in held-out test set

plt.style.use('ggplot')

# Plot the smoothed calibration curve with confidence bands and rug plot
fig, ax = plt.subplots(figsize=(6, 5))

ax.grid(False)
ax.set_facecolor("white")

for spine in ["bottom", "left"]:
    ax.spines[spine].set_color("black")
    ax.spines[spine].set_linewidth(1)

ax.plot([0, 1], [0, 1], "--", color="grey", linewidth=1, label="Ideal")
ax.plot(grid, fit, color="navy", linewidth=1.3, label="Smoothed calibration curve")
ax.fill_between(grid, lo, hi, color="grey", alpha=0.33, label="95% confidence interval")

ax.set_xlabel("Predicted probability", fontsize=10)
ax.set_ylabel("Observed probability", fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xticks([0.2, 0.4, 0.6, 0.8, 1])
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1])
ax.tick_params(axis="both", labelsize=8)

# rug plot parameters
bins = 100 # adjustable
counts, edges = np.histogram(y_prob, bins=bins, range=(0, 1)) # creates two arrays
centrepoints = 0.5 * (edges[:-1] + edges[1:])
rug_frac = 0.15 # scale relative to y axis
heights = (counts / counts.max()) * rug_frac 

ax.vlines(
    centrepoints, 0, heights,
    transform=ax.get_xaxis_transform(),  # x in data coordinates, y in axis coordinates
    color="black", alpha=0.3, linewidth=1,
    clip_on=False
)

ax.legend(fontsize=8)
#plt.show()
plt.savefig(f"{PREFIX}_testset_calibration_curve.tiff", bbox_inches="tight", dpi=600)

#--------------------------------------------------------------------------------------

# Plot the quantile calibration plot
n_bins = 10
alpha = 0.05
df = pd.DataFrame({
    "y_true": np.asarray(y_test),
    "pred_prob": np.asarray(y_prob)
})
n_unique_pred_probs = df["pred_prob"].nunique()

if n_unique_pred_probs <= n_bins:
    # For models with few probability outputs, group directly by the exact predicted probability.
    df["prob_bin"] = df["pred_prob"]

else:
    # Otherwise use deciles of predicted probability.
    df["prob_bin"] = pd.qcut(
        df["pred_prob"],
        q=n_bins,
        duplicates="drop"
    )

## Summarise calibration per bin
calibration = (
    df
    .groupby("prob_bin", observed=True)
    .agg(
        n=("y_true", "size"),
        events=("y_true", "sum"),
        mean_pred_prob=("pred_prob", "mean"),
        min_pred_prob=("pred_prob", "min"),
        max_pred_prob=("pred_prob", "max")
    )
    .reset_index()
)
calibration["observed_prob"] = calibration["events"] / calibration["n"]

## Calculate Wilson confidence intervals for observed probabilities
ci_values = [
    wilson_ci(k=row["events"], n=row["n"], alpha=alpha)
    for _, row in calibration.iterrows()
]

calibration["ci_lower"] = [ci[0] for ci in ci_values]
calibration["ci_upper"] = [ci[1] for ci in ci_values]

calibration = calibration.sort_values("mean_pred_prob").reset_index(drop=True) # Sort predicted probabilities

## Plot
fig, ax = plt.subplots(figsize=(6, 5))

ax.grid(False)
ax.set_facecolor("white")

for spine in ["bottom", "left"]:
    ax.spines[spine].set_color("black")
    ax.spines[spine].set_linewidth(1)

x_plot = calibration["mean_pred_prob"].to_numpy()
y_plot = calibration["observed_prob"].to_numpy()

yerr_lower = y_plot - calibration["ci_lower"].to_numpy()
yerr_upper = calibration["ci_upper"].to_numpy() - y_plot

ax.errorbar(
    x_plot,
    y_plot,
    yerr=np.vstack([yerr_lower, yerr_upper]),
    fmt="o",
    capsize=3,
    markersize=5,
    color="darkolivegreen",
    linestyle="none"
)

ax.plot([0, 1], [0, 1], "--", color="grey", linewidth=1, label="Ideal calibration")
ax.set_xlabel("Mean predicted probability", fontsize=10)
ax.set_ylabel("Observed probability", fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xticks([0.2, 0.4, 0.6, 0.8, 1])
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1])
ax.tick_params(axis="both", labelsize=8)

#plt.show()
plt.savefig(f"{PREFIX}_testset_decile_calibration_plot.tiff", bbox_inches="tight", dpi=600)


#--------------------------------------------------------------------------------------

# Plot the ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)  
auc = roc_auc_score(y_test, y_prob)      

fig, ax = plt.subplots(figsize=(6, 5))

ax.grid(False)
ax.set_facecolor("white")

for spine in ["bottom", "left"]:
    ax.spines[spine].set_color("black")
    ax.spines[spine].set_linewidth(1)
    
ax.plot(fpr, tpr, linewidth=1.3, color="darkred", label=f"AUC={auc:.3f}")
ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1, label="Chance")
ax.set_xlabel("False positive rate", fontsize=10)
ax.set_ylabel("True positive rate", fontsize=10)
ax.set_xlim(0, 1)
ax.set_xticks([0.2, 0.4, 0.6, 0.8, 1])
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1])
ax.set_ylim(0, 1)
ax.tick_params(axis="both", labelsize=8)
ax.annotate(
    f"AUC: {auc:.3f}",
    xy=(0.05, 0.9), 
    fontsize=10,
    alpha=0.75
)
#plt.show()
plt.savefig(f"{PREFIX}_testset_roc_curve.tiff", bbox_inches="tight", dpi=600)
#--------------------------------------------------------------------------------------

# Plot precision recall curve
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
auprc = average_precision_score(y_test, y_prob)

fig, ax = plt.subplots(figsize=(6, 5))

ax.grid(False)
ax.set_facecolor("white")
for spine in ["bottom", "left"]:
    ax.spines[spine].set_color("black")
    ax.spines[spine].set_linewidth(1)

ax.plot(
    recall,
    precision,
    color="#B45309",
    linewidth=1.1,
)

ax.set_xlabel("Recall (sensitivity)", fontsize=10)
ax.set_ylabel("Precision (PPV)", fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xticks([0.2, 0.4, 0.6, 0.8, 1])
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1])
ax.tick_params(axis="both", labelsize=8)
ax.annotate(
    f"AUPRC: {auprc:.3f}",
    xy=(0.05, 0.5), 
    fontsize=10,
    alpha=0.75
)

#plt.show()
plt.savefig(f"{PREFIX}_testset_pr_curve.tiff", bbox_inches="tight", dpi=600)

In [ ]:
result = cluster_bootstrap_cis(y_test, y_prob, subject_id)
result

df_ci = (
    pd.DataFrame.from_dict(result, orient="index", columns=["score", "ci_low", "ci_high"])
      .reset_index()
      .rename(columns={"index": "metric"})
)

oe = oe_ratio_ci_cluster_bootstrap(y_test, y_prob, subject_id)

cal = {**calib_metrics, **oe}

rows = []
for k, v in cal.items():
    if k.endswith("_ci_95"):
        continue
    ci_low, ci_high = cal.get(f"{k}_ci_95", (np.nan, np.nan))
    rows.append({"metric": k, "score": v, "ci_low": ci_low, "ci_high": ci_high})


cal_df = pd.DataFrame(rows, columns=["metric", "score", "ci_low", "ci_high"])

metrics_df = pd.concat([df_ci, cal_df], ignore_index=True)

metrics_df.to_csv(f"{PREFIX}_testset_metrics.csv")

metrics_df

### Assess feature importance
- Permutation importance
- SHAP plot

In [ ]:
# Calculate feature permutation importances
# NB: Here the permuation importance is the drop in AUROC when the variable is permuted

## Handle columns that never entered the pipeline
expected = list(final_model.feature_names_in_)
extras = [c for c in X_test.columns if c not in expected]

X_test_aligned = X_test.drop(columns=extras)[expected]

result = permutation_importance(
    final_model,        
    X_test_aligned,
    y_test,     
    n_repeats=200,
    random_state=12,
    scoring="roc_auc",  
    n_jobs=-1
)

In [ ]:
# Build a readable DataFrame
pi_df = (pd.DataFrame({
            "feature": (X_test_aligned.columns).map(feature_dict),
            "importance_mean": result.importances_mean,
            "importance_std": result.importances_std
        })
        .sort_values("importance_mean", ascending=False)
        .reset_index(drop=True))


stats_df = (pd.DataFrame({
    "feature": (X_test_aligned.columns).map(feature_dict),
    "median": np.median(result.importances, axis=1),
    "p2_5":   np.percentile(result.importances,  2.5, axis=1),
    "p97_5":  np.percentile(result.importances, 97.5, axis=1),
}).sort_values("median", ascending=False)
    .reset_index(drop=True))

# Save and print top features
print(stats_df.head(20).to_string(index=False))
stats_df.to_csv(f"{PREFIX}_pi_auc.csv", index=False)

# Plot top n importances
top_n = 20
plot_df = stats_df.head(top_n).iloc[::-1].copy()

x = plot_df["median"].to_numpy()
lo = plot_df["p2_5"].to_numpy()
hi = plot_df["p97_5"].to_numpy()
xerr = np.vstack([x - lo, hi - x])

y = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(x,
            y, 
            xerr=xerr, 
            fmt="o", 
            markersize=6, 
            capsize=3, 
            elinewidth=1.3,
            color="darkred")

ax.set_facecolor("white")
ax.grid(True, axis="x", color="lightgrey", linestyle="-", linewidth=0.8, alpha=0.8)
ax.grid(False, axis="y")
ax.spines["bottom"].set_visible(True)
ax.spines["bottom"].set_color("black")
ax.spines["left"].set_visible(True)
ax.spines["left"].set_color("black")
ax.set_xlim(0, 0.08)
ax.set_yticks(y)
ax.tick_params(labelsize=8)
ax.tick_params(left=False)
ax.set_xticks(np.arange(0.02, 0.07, 0.02))
ax.set_yticklabels(plot_df["feature"])
ax.set_xlabel("Permutation importance", fontsize=10)
ax.set_ylabel("")
ax.axvline(0, color="lightgrey", linewidth=1)
plt.tight_layout()
plt.savefig(f"{PREFIX}_permutation_importances.tiff", dpi=600, bbox_inches="tight")

#plt.show()

In [ ]:
# SHAP values and plotting

background_size = 500
nsamples = 1000
random_state = 12
positive_class = 1 
X_train = X

# Helper functions
def select_rows(X, idx):
    if hasattr(X, "iloc"):
        return X.iloc[idx]
    return X[idx]


def to_dense(X):
    """
    Convert sparse matrix to dense numpy array if needed.
    """
    if hasattr(X, "toarray"):
        return X.toarray()
    return np.asarray(X)


# Separate preprocessing pipeline from final model
preprocessor = final_model[:-1]
model = final_model[-1]


# Select background rows from X_train
rng = np.random.default_rng(random_state)
n_train = X_train.shape[0]
background_n = min(background_size, n_train)

background_idx = rng.choice(
    np.arange(n_train),
    size=background_n,
    replace=False
)

X_background_raw = select_rows(X_train, background_idx)


# Transform background and all of X_test
background = preprocessor.transform(X_background_raw)
X_test_transformed = preprocessor.transform(X_test)

background = to_dense(background)
X_test_transformed = to_dense(X_test_transformed)


# Extract and clean feature names
try:
    feature_names = list(preprocessor.get_feature_names_out())
except Exception:
    feature_names = [f"feature_{i}" for i in range(X_test_transformed.shape[1])]

try:
    names_cleaned = [x.replace("_1", "") for x in feature_names]
    mapped_names = [feature_dict.get(x, x) for x in names_cleaned]
except Exception:
    mapped_names = feature_names


# Safety checks
assert background.ndim == 2, "Background data must be 2D."
assert X_test_transformed.ndim == 2, "X_test_transformed must be 2D."

assert background.shape[1] == X_test_transformed.shape[1], (
    f"Background has {background.shape[1]} columns, "
    f"X_test_transformed has {X_test_transformed.shape[1]} columns."
)

assert X_test_transformed.shape[1] == len(mapped_names), (
    f"Transformed data has {X_test_transformed.shape[1]} columns, "
    f"mapped_names has {len(mapped_names)} names."
)


# Define prediction function for KernelExplainer
def predict_positive_class(X):
    """
    KernelExplainer passes transformed data into this function.
    NB the model is the final estimator, not the full pipeline.
    """
    X = to_dense(X)

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)

        if np.asarray(proba).ndim == 2:
            return proba[:, positive_class]

        return proba

    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X)

        if np.asarray(scores).ndim == 2:
            return scores[:, positive_class]

        return scores

    else:
        raise AttributeError(
            "Model must have either predict_proba or decision_function."
        )


# Test prediction function
test_pred = predict_positive_class(X_test_transformed[:5])

assert np.asarray(test_pred).ndim == 1, (
    "Prediction function must return a 1D array."
)


# Create Kernel SHAP explainer

explainer = shap.KernelExplainer(
    predict_positive_class,
    background
)

# Calculate SHAP values for X_test

shap_values = explainer.shap_values(
    X_test_transformed,
    nsamples=nsamples
)

if isinstance(shap_values, list):
    shap_values_to_plot = shap_values[0]
else:
    shap_values_to_plot = np.asarray(shap_values)


# Safety checks

assert shap_values_to_plot.ndim == 2, (
    f"Expected 2D SHAP values, got shape {shap_values_to_plot.shape}."
)

assert shap_values_to_plot.shape == X_test_transformed.shape, (
    f"SHAP shape mismatch: SHAP values are {shap_values_to_plot.shape}, "
    f"X_test_transformed is {X_test_transformed.shape}."
)

np.save(f"{PREFIX}_SHAP_values_to_plot.npy", shap_values_to_plot)

In [ ]:
# Beeswarm plot
fig, ax = plt.subplots(figsize=(5, 9))

shap.summary_plot(
    shap_values_to_plot,
    X_test_transformed,
    feature_names=mapped_names,
    show=False,
    max_display=20
)

ax = plt.gca()
ax.grid(False)
ax.set_facecolor("white")

for spine in ["bottom", "left"]:
    ax.spines[spine].set_color("black")
    ax.spines[spine].set_linewidth(1)

ax.tick_params(axis="both", labelsize=9)
ax.set_xlabel("SHAP value (impact on model output)", fontsize=10)
ax.set_xlim(-0.4, 0.4)
fig = plt.gcf()

if len(fig.axes) > 1:
    cbar_ax = fig.axes[-1]
    cbar_ax.tick_params(labelsize=10)
    cbar_ax.set_ylabel("Feature value", fontsize=10)

plt.savefig(f"{PREFIX}_SHAP_plot.tiff", dpi=600, bbox_inches="tight")
#plt.show()